# verify

> Verification route that queries graph and computes results

In [ ]:
#| default_exp routes.verify

In [ ]:
#| export
from typing import Dict, Callable, Tuple

from fasthtml.common import APIRouter

from cjm_transcript_verify.models import VerifyUrls
from cjm_transcript_verify.services.verify import VerifyService
from cjm_transcript_verify.routes.core import WorkflowStateStore, _load_verify_context
from cjm_transcript_verify.components.step_renderer import render_verify_step

# Debug flag
DEBUG_VERIFY_ROUTES = False

## Router Initialization

In [ ]:
#| export
def init_verify_router(
    state_store:WorkflowStateStore,  # The workflow state store
    workflow_id:str,  # The workflow identifier
    prefix:str,  # Route prefix (e.g., "/workflow/verify")
    verify_service:VerifyService,  # Service for graph queries
    urls:VerifyUrls,  # URL bundle (will be populated)
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, routes dict)
    """Initialize verify route that computes verification results."""
    router = APIRouter(prefix=prefix)
    routes = {}
    
    @router
    async def verify(request, sess):
        """Compute and return verification dashboard."""
        session_id = sess.get("session_id", "default")
        
        if DEBUG_VERIFY_ROUTES:
            print(f"[VERIFY_ROUTES] verify called")
            print(f"[VERIFY_ROUTES] session_id: {session_id}")
        
        # Load context to get document_id
        ctx = _load_verify_context(state_store, workflow_id, session_id)
        
        if not ctx.document_id:
            if DEBUG_VERIFY_ROUTES:
                print(f"[VERIFY_ROUTES] No document_id found")
            return render_verify_step(
                result=None,
                urls=urls,
                error="No document ID found. Please complete the Review step first."
            )
        
        # Query graph and compute verification
        result = await verify_service.verify_document_async(ctx.document_id)
        
        if result is None:
            if DEBUG_VERIFY_ROUTES:
                print(f"[VERIFY_ROUTES] verify_document returned None")
            return render_verify_step(
                result=None,
                urls=urls,
                error=f"Could not verify document {ctx.document_id[:12]}..."
            )
        
        if DEBUG_VERIFY_ROUTES:
            print(f"[VERIFY_ROUTES] Verification complete: {result.segment_count} segments")
        
        return render_verify_step(result=result, urls=urls)
    
    routes["verify"] = verify
    
    return router, routes

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()